In [1]:

#!pip install statsmodels
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

In [ ]:
###Data without pre-processing steps

#1st regresion model for data without pre-processing steps - no weight

#loading data
data_without_prep = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/final_ihd_dataset_without_prepocessing.csv")    

#checking the row counts
print(data_without_prep['gender'].value_counts())

#deleting all the null values from the dataset
data_without_prep = data_without_prep.dropna(subset=['year','cholesterol_level','diabetes_level','hypertension_level','smoking_level','obesity_level'])

#checking the row counts after deleting null values
print(data_without_prep['gender'].value_counts())

#extracting features and target variable
features = [
    'cholesterol_level',
    'diabetes_level', 
    'hypertension_level',
    'smoking_level',
    'obesity_level',
    'year'          
]

X = sm.add_constant(data_without_prep[features])
y = data_without_prep['incidence_rate']
gender = data_without_prep['gender']

# repeated 5-fold CV
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

#lists for results
mse_l, rmse_l, mae_l, r2_l = [], [], [], []
pred_f_l, pred_m_l = [], []
resid_f_l, resid_m_l = [], []
rmse_f_l, rmse_m_l = [], []


#training
for train_idx, test_idx in rkf.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    gender_te = gender.iloc[test_idx]
    
    #model fitting
    model = sm.OLS(y_tr, X_tr).fit()
    y_pred = model.predict(X_te)
    
    #performance
    mse_l.append(mean_squared_error(y_te, y_pred))
    rmse_l.append(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae_l.append(mean_absolute_error(y_te, y_pred))
    r2_l.append(r2_score(y_te, y_pred))

    #fairness
    f_mask = gender_te == 'Female'
    m_mask = gender_te == 'Male'
    
    pred_f_l.append(y_pred[f_mask].mean() if f_mask.any() else np.nan)
    pred_m_l.append(y_pred[m_mask].mean() if m_mask.any() else np.nan)
    resid_f_l.append((y_te[f_mask] - y_pred[f_mask]).mean() if f_mask.any() else np.nan)
    resid_m_l.append((y_te[m_mask] - y_pred[m_mask]).mean() if m_mask.any() else np.nan)
    rmse_f_l.append(np.sqrt(mean_squared_error(y_te[f_mask], y_pred[f_mask])) if f_mask.any() else np.nan)
    rmse_m_l.append(np.sqrt(mean_squared_error(y_te[m_mask], y_pred[m_mask])) if m_mask.any() else np.nan)

#results display
print("\n" + "="*65)
print("BASELINE MODEL – NO WEIGHTS (Repeated 5-fold CV, 50 folds)".center(65))
print("="*65)
print(f"MSE               : {np.mean(mse_l):8.1f} ± {np.std(mse_l):.1f}")
print(f"RMSE              : {np.mean(rmse_l):8.1f} ± {np.std(rmse_l):.1f}")
print(f"MAE               : {np.mean(mae_l):8.1f} ± {np.std(mae_l):.1f}")
print(f"R²                : {np.mean(r2_l):8.3f} ± {np.std(r2_l):.3f}")

print(f"\nMean predicted risk – Women    : {np.nanmean(pred_f_l):6.1f}")
print(f"Mean predicted risk – Men      : {np.nanmean(pred_m_l):6.1f}")
print(f"Mean residual (Y − Ŷ) – Women  : {np.nanmean(resid_f_l):+7.2f}")
print(f"Mean residual (Y − Ŷ) – Men    : {np.nanmean(resid_m_l):+7.2f}")
print(f"RMSE – Women                   : {np.nanmean(rmse_f_l):6.1f}")
print(f"RMSE – Men                     : {np.nanmean(rmse_m_l):6.1f}")
print("="*65)


gender
Male      722
Female    722
Name: count, dtype: int64
gender
Male      209
Female    209
Name: count, dtype: int64

    BASELINE MODEL – NO WEIGHTS (Repeated 5-fold CV, 50 folds)   
MSE               :  14448.3 ± 0.0
RMSE              :    120.2 ± 0.0
MAE               :     92.1 ± 0.0
R²                :    0.582 ± 0.000

Mean predicted risk – Women    :  294.8
Mean predicted risk – Men      :  430.4
Mean residual (Y − Ŷ) – Women  :   +1.40
Mean residual (Y − Ŷ) – Men    :   +2.96
RMSE – Women                   :   83.8
RMSE – Men                     :  138.8

    BASELINE MODEL – NO WEIGHTS (Repeated 5-fold CV, 50 folds)   
MSE               :  15320.3 ± 872.0
RMSE              :    123.7 ± 3.5
MAE               :     97.0 ± 4.9
R²                :    0.472 ± 0.109

Mean predicted risk – Women    :  305.3
Mean predicted risk – Men      :  403.0
Mean residual (Y − Ŷ) – Women  :  -12.61
Mean residual (Y − Ŷ) – Men    :   +8.10
RMSE – Women                   :   93.2
RMSE – Men  

In [ ]:
#2nd regresion model for data without pre-processing steps - weight 35% female
#loading data
data_without_prep = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/final_ihd_dataset_without_prepocessing.csv")    

#checking the row counts
print(data_without_prep['gender'].value_counts())

#deleting all the null values from the dataset
data_without_prep = data_without_prep.dropna(subset=['year','cholesterol_level','diabetes_level','hypertension_level','smoking_level','obesity_level', 'weight_35f'])

#checking the row counts after deleting null values
print(data_without_prep['gender'].value_counts())

#extracting features and target variable
features = [
    'cholesterol_level',
    'diabetes_level', 
    'hypertension_level',
    'smoking_level',
    'obesity_level',
    'year'          
]

X = sm.add_constant(data_without_prep[features])
y = data_without_prep['incidence_rate']
gender = data_without_prep['gender']
weights_35f = data_without_prep['weight_35f'].values

# repeated 5-fold CV
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

#lists for results
mse_l, rmse_l, mae_l, r2_l = [], [], [], []
pred_f_l, pred_m_l = [], []
resid_f_l, resid_m_l = [], []
rmse_f_l, rmse_m_l = [], []


#training
for train_idx, test_idx in rkf.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    gender_te = gender.iloc[test_idx]
    w_tr = weights_35f[train_idx]
    
    #model fitting
    model = sm.WLS(y_tr, X_tr, weights=w_tr).fit()
    y_pred = model.predict(X_te)
    
    #performance
    mse_l.append(mean_squared_error(y_te, y_pred))
    rmse_l.append(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae_l.append(mean_absolute_error(y_te, y_pred))
    r2_l.append(r2_score(y_te, y_pred))

    #fairness
    f_mask = gender_te == 'Female'
    m_mask = gender_te == 'Male'
    
    pred_f_l.append(y_pred[f_mask].mean() if f_mask.any() else np.nan)
    pred_m_l.append(y_pred[m_mask].mean() if m_mask.any() else np.nan)
    resid_f_l.append((y_te[f_mask] - y_pred[f_mask]).mean() if f_mask.any() else np.nan)
    resid_m_l.append((y_te[m_mask] - y_pred[m_mask]).mean() if m_mask.any() else np.nan)
    rmse_f_l.append(np.sqrt(mean_squared_error(y_te[f_mask], y_pred[f_mask])) if f_mask.any() else np.nan)
    rmse_m_l.append(np.sqrt(mean_squared_error(y_te[m_mask], y_pred[m_mask])) if m_mask.any() else np.nan)

#results display
print("\n" + "="*65)
print("BASELINE MODEL – WEIGHT 35% FEMALE (Repeated 5-fold CV, 50 folds)".center(65))
print("="*65)
print(f"MSE               : {np.mean(mse_l):8.1f} ± {np.std(mse_l):.1f}")
print(f"RMSE              : {np.mean(rmse_l):8.1f} ± {np.std(rmse_l):.1f}")
print(f"MAE               : {np.mean(mae_l):8.1f} ± {np.std(mae_l):.1f}")
print(f"R²                : {np.mean(r2_l):8.3f} ± {np.std(r2_l):.3f}")

print(f"\nMean predicted risk – Women    : {np.nanmean(pred_f_l):6.1f}")
print(f"Mean predicted risk – Men      : {np.nanmean(pred_m_l):6.1f}")
print(f"Mean residual (Y − Ŷ) – Women  : {np.nanmean(resid_f_l):+7.2f}")
print(f"Mean residual (Y − Ŷ) – Men    : {np.nanmean(resid_m_l):+7.2f}")
print(f"RMSE – Women                   : {np.nanmean(rmse_f_l):6.1f}")
print(f"RMSE – Men                     : {np.nanmean(rmse_m_l):6.1f}")
print("="*65)

gender
Male      722
Female    722
Name: count, dtype: int64
gender
Male      209
Female    209
Name: count, dtype: int64

BASELINE MODEL – WEIGHT 35% FEMALE (Repeated 5-fold CV, 50 folds)
MSE               :  14507.2 ± 0.0
RMSE              :    120.4 ± 0.0
MAE               :     92.5 ± 0.0
R²                :    0.580 ± 0.000

Mean predicted risk – Women    :  296.6
Mean predicted risk – Men      :  431.7
Mean residual (Y − Ŷ) – Women  :   -0.46
Mean residual (Y − Ŷ) – Men    :   +1.72
RMSE – Women                   :   85.3
RMSE – Men                     :  138.5

BASELINE MODEL – WEIGHT 35% FEMALE (Repeated 5-fold CV, 50 folds)
MSE               :  15323.5 ± 816.3
RMSE              :    123.7 ± 3.3
MAE               :     97.1 ± 4.6
R²                :    0.472 ± 0.107

Mean predicted risk – Women    :  306.8
Mean predicted risk – Men      :  404.3
Mean residual (Y − Ŷ) – Women  :  -14.17
Mean residual (Y − Ŷ) – Men    :   +6.82
RMSE – Women                   :   94.2
RMSE – Men  

In [ ]:
#3rd regresion model for data without pre-processing steps - weight 30 % female

#loading data
data_without_prep = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/final_ihd_dataset_without_prepocessing.csv")    

#checking the row counts
print(data_without_prep['gender'].value_counts())

#deleting all the null values from the dataset
data_without_prep = data_without_prep.dropna(subset=['year','cholesterol_level','diabetes_level','hypertension_level','smoking_level','obesity_level', 'weight_30f'])

#checking the row counts after deleting null values
print(data_without_prep['gender'].value_counts())

#extracting features and target variable
features = [
    'cholesterol_level',
    'diabetes_level', 
    'hypertension_level',
    'smoking_level',
    'obesity_level',
    'year'          
]

X = sm.add_constant(data_without_prep[features])
y = data_without_prep['incidence_rate']
gender = data_without_prep['gender']
weights_30f = data_without_prep['weight_30f'].values

# repeated 5-fold CV
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

#lists for results
mse_l, rmse_l, mae_l, r2_l = [], [], [], []
pred_f_l, pred_m_l = [], []
resid_f_l, resid_m_l = [], []
rmse_f_l, rmse_m_l = [], []


#training
for train_idx, test_idx in rkf.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    gender_te = gender.iloc[test_idx]
    w_tr = weights_30f[train_idx]
    
    #model fitting
    model = sm.WLS(y_tr, X_tr, weights=w_tr).fit()
    y_pred = model.predict(X_te)
    
    #performance
    mse_l.append(mean_squared_error(y_te, y_pred))
    rmse_l.append(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae_l.append(mean_absolute_error(y_te, y_pred))
    r2_l.append(r2_score(y_te, y_pred))

    #fairness
    f_mask = gender_te == 'Female'
    m_mask = gender_te == 'Male'
    
    pred_f_l.append(y_pred[f_mask].mean() if f_mask.any() else np.nan)
    pred_m_l.append(y_pred[m_mask].mean() if m_mask.any() else np.nan)
    resid_f_l.append((y_te[f_mask] - y_pred[f_mask]).mean() if f_mask.any() else np.nan)
    resid_m_l.append((y_te[m_mask] - y_pred[m_mask]).mean() if m_mask.any() else np.nan)
    rmse_f_l.append(np.sqrt(mean_squared_error(y_te[f_mask], y_pred[f_mask])) if f_mask.any() else np.nan)
    rmse_m_l.append(np.sqrt(mean_squared_error(y_te[m_mask], y_pred[m_mask])) if m_mask.any() else np.nan)

#results display
print("\n" + "="*65)
print("BASELINE MODEL – WEIGHT 30% FEMALE (Repeated 5-fold CV, 50 folds)".center(65))
print("="*65)
print(f"MSE               : {np.mean(mse_l):8.1f} ± {np.std(mse_l):.1f}")
print(f"RMSE              : {np.mean(rmse_l):8.1f} ± {np.std(rmse_l):.1f}")
print(f"MAE               : {np.mean(mae_l):8.1f} ± {np.std(mae_l):.1f}")
print(f"R²                : {np.mean(r2_l):8.3f} ± {np.std(r2_l):.3f}")

print(f"\nMean predicted risk – Women    : {np.nanmean(pred_f_l):6.1f}")
print(f"Mean predicted risk – Men      : {np.nanmean(pred_m_l):6.1f}")
print(f"Mean residual (Y − Ŷ) – Women  : {np.nanmean(resid_f_l):+7.2f}")
print(f"Mean residual (Y − Ŷ) – Men    : {np.nanmean(resid_m_l):+7.2f}")
print(f"RMSE – Women                   : {np.nanmean(rmse_f_l):6.1f}")
print(f"RMSE – Men                     : {np.nanmean(rmse_m_l):6.1f}")
print("="*65)


gender
Male      722
Female    722
Name: count, dtype: int64
gender
Male      209
Female    209
Name: count, dtype: int64

BASELINE MODEL – WEIGHT 30% FEMALE (Repeated 5-fold CV, 50 folds)
MSE               :  14628.2 ± 0.0
RMSE              :    120.9 ± 0.0
MAE               :     93.0 ± 0.0
R²                :    0.576 ± 0.000

Mean predicted risk – Women    :  299.6
Mean predicted risk – Men      :  433.5
Mean residual (Y − Ŷ) – Women  :   -3.48
Mean residual (Y − Ŷ) – Men    :   -0.09
RMSE – Women                   :   87.8
RMSE – Men                     :  138.2

BASELINE MODEL – WEIGHT 30% FEMALE (Repeated 5-fold CV, 50 folds)
MSE               :  15357.9 ± 729.7
RMSE              :    123.9 ± 2.9
MAE               :     97.2 ± 4.2
R²                :    0.472 ± 0.105

Mean predicted risk – Women    :  309.4
Mean predicted risk – Men      :  406.2
Mean residual (Y − Ŷ) – Women  :  -16.78
Mean residual (Y − Ŷ) – Men    :   +4.93
RMSE – Women                   :   95.9
RMSE – Men  

In [ ]:
#4th regresion model for data without pre-processing steps - weight 25 % female

#loading data
data_without_prep = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/final_ihd_dataset_without_prepocessing.csv")    

#checking the row counts
print(data_without_prep['gender'].value_counts())

#deleting all the null values from the dataset
data_without_prep = data_without_prep.dropna(subset=['year','cholesterol_level','diabetes_level','hypertension_level','smoking_level','obesity_level', 'weight_25f'])

#checking the row counts after deleting null values
print(data_without_prep['gender'].value_counts())

#extracting features and target variable
features = [
    'cholesterol_level',
    'diabetes_level', 
    'hypertension_level',
    'smoking_level',
    'obesity_level',
    'year'          
]

X = sm.add_constant(data_without_prep[features])
y = data_without_prep['incidence_rate']
gender = data_without_prep['gender']
weights_25f = data_without_prep['weight_25f'].values

# repeated 5-fold CV
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

#lists for results
mse_l, rmse_l, mae_l, r2_l = [], [], [], []
pred_f_l, pred_m_l = [], []
resid_f_l, resid_m_l = [], []
rmse_f_l, rmse_m_l = [], []


#training
for train_idx, test_idx in rkf.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    gender_te = gender.iloc[test_idx]
    w_tr = weights_25f[train_idx]
    
    #model fitting
    model = sm.WLS(y_tr, X_tr, weights=w_tr).fit()
    y_pred = model.predict(X_te)
    
    #performance
    mse_l.append(mean_squared_error(y_te, y_pred))
    rmse_l.append(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae_l.append(mean_absolute_error(y_te, y_pred))
    r2_l.append(r2_score(y_te, y_pred))

    #fairness
    f_mask = gender_te == 'Female'
    m_mask = gender_te == 'Male'
    
    pred_f_l.append(y_pred[f_mask].mean() if f_mask.any() else np.nan)
    pred_m_l.append(y_pred[m_mask].mean() if m_mask.any() else np.nan)
    resid_f_l.append((y_te[f_mask] - y_pred[f_mask]).mean() if f_mask.any() else np.nan)
    resid_m_l.append((y_te[m_mask] - y_pred[m_mask]).mean() if m_mask.any() else np.nan)
    rmse_f_l.append(np.sqrt(mean_squared_error(y_te[f_mask], y_pred[f_mask])) if f_mask.any() else np.nan)
    rmse_m_l.append(np.sqrt(mean_squared_error(y_te[m_mask], y_pred[m_mask])) if m_mask.any() else np.nan)

#results display
print("\n" + "="*65)
print("BASELINE MODEL – WEIGHT 25% FEMALE (Repeated 5-fold CV, 50 folds)".center(65))
print("="*65)
print(f"MSE               : {np.mean(mse_l):8.1f} ± {np.std(mse_l):.1f}")
print(f"RMSE              : {np.mean(rmse_l):8.1f} ± {np.std(rmse_l):.1f}")
print(f"MAE               : {np.mean(mae_l):8.1f} ± {np.std(mae_l):.1f}")
print(f"R²                : {np.mean(r2_l):8.3f} ± {np.std(r2_l):.3f}")

print(f"\nMean predicted risk – Women    : {np.nanmean(pred_f_l):6.1f}")
print(f"Mean predicted risk – Men      : {np.nanmean(pred_m_l):6.1f}")
print(f"Mean residual (Y − Ŷ) – Women  : {np.nanmean(resid_f_l):+7.2f}")
print(f"Mean residual (Y − Ŷ) – Men    : {np.nanmean(resid_m_l):+7.2f}")
print(f"RMSE – Women                   : {np.nanmean(rmse_f_l):6.1f}")
print(f"RMSE – Men                     : {np.nanmean(rmse_m_l):6.1f}")
print("="*65)


gender
Male      722
Female    722
Name: count, dtype: int64
gender
Male      209
Female    209
Name: count, dtype: int64

BASELINE MODEL – WEIGHT 25% FEMALE (Repeated 5-fold CV, 50 folds)
MSE               :  14808.3 ± 0.0
RMSE              :    121.7 ± 0.0
MAE               :     93.6 ± 0.0
R²                :    0.571 ± 0.000

Mean predicted risk – Women    :  303.3
Mean predicted risk – Men      :  435.4
Mean residual (Y − Ŷ) – Women  :   -7.12
Mean residual (Y − Ŷ) – Men    :   -2.05
RMSE – Women                   :   90.8
RMSE – Men                     :  138.0

BASELINE MODEL – WEIGHT 25% FEMALE (Repeated 5-fold CV, 50 folds)
MSE               :  15440.3 ± 631.9
RMSE              :    124.2 ± 2.5
MAE               :     97.4 ± 3.8
R²                :    0.469 ± 0.102

Mean predicted risk – Women    :  312.7
Mean predicted risk – Men      :  408.3
Mean residual (Y − Ŷ) – Women  :  -20.03
Mean residual (Y − Ŷ) – Men    :   +2.85
RMSE – Women                   :   98.1
RMSE – Men  

In [11]:
###Data with pre-processing steps

#1st regresion model for data with  pre-processing steps - no weight


#loading data
data_with_prep = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/final_ihd_dataset_with_prepocessing.csv")    

#checking the row counts
print(data_with_prep['gender'].value_counts())

#deleting all the null values from the dataset
data_with_prep = data_with_prep.dropna(subset=['incidence_rate','year','cholesterol_level','diabetes_level','hypertension_level','smoking_level','obesity_level','alcohol_consumption', 'health_expenditure'])

#checking the row counts after deleting null values
print(data_with_prep['gender'].value_counts())

#extracting features and target variable
features = [
    'cholesterol_level',
    'diabetes_level', 
    'hypertension_level',
    'smoking_level',
    'obesity_level',
    'alcohol_consumption', 
    'health_expenditure',
    'year'          
]

X = sm.add_constant(data_with_prep[features])
y = data_with_prep['incidence_rate']
gender = data_with_prep['gender']

# repeated 5-fold CV
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

#lists for results
mse_l, rmse_l, mae_l, r2_l = [], [], [], []
pred_f_l, pred_m_l = [], []
resid_f_l, resid_m_l = [], []
rmse_f_l, rmse_m_l = [], []


#training
for train_idx, test_idx in rkf.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    gender_te = gender.iloc[test_idx]
    
    #model fitting
    model = sm.OLS(y_tr, X_tr).fit()
    y_pred = model.predict(X_te)
    
    #performance
    mse_l.append(mean_squared_error(y_te, y_pred))
    rmse_l.append(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae_l.append(mean_absolute_error(y_te, y_pred))
    r2_l.append(r2_score(y_te, y_pred))

    #fairness
    f_mask = gender_te == 'Female'
    m_mask = gender_te == 'Male'
    
    pred_f_l.append(y_pred[f_mask].mean() if f_mask.any() else np.nan)
    pred_m_l.append(y_pred[m_mask].mean() if m_mask.any() else np.nan)
    resid_f_l.append((y_te[f_mask] - y_pred[f_mask]).mean() if f_mask.any() else np.nan)
    resid_m_l.append((y_te[m_mask] - y_pred[m_mask]).mean() if m_mask.any() else np.nan)
    rmse_f_l.append(np.sqrt(mean_squared_error(y_te[f_mask], y_pred[f_mask])) if f_mask.any() else np.nan)
    rmse_m_l.append(np.sqrt(mean_squared_error(y_te[m_mask], y_pred[m_mask])) if m_mask.any() else np.nan)

#results display
print("\n" + "="*65)
print("PREPROCESSED MODEL – NO WEIGHTS (Repeated 5-fold CV, 50 folds)".center(65))
print("="*65)
print(f"MSE               : {np.mean(mse_l):8.1f} ± {np.std(mse_l):.1f}")
print(f"RMSE              : {np.mean(rmse_l):8.1f} ± {np.std(rmse_l):.1f}")
print(f"MAE               : {np.mean(mae_l):8.1f} ± {np.std(mae_l):.1f}")
print(f"R²                : {np.mean(r2_l):8.3f} ± {np.std(r2_l):.3f}")

print(f"\nMean predicted risk – Women  : {np.nanmean(pred_f_l):6.1f}")
print(f"Mean predicted risk – Men      : {np.nanmean(pred_m_l):6.1f}")
print(f"Mean residual (Y − Ŷ) – Women  : {np.nanmean(resid_f_l):+7.2f}")
print(f"Mean residual (Y − Ŷ) – Men    : {np.nanmean(resid_m_l):+7.2f}")
print(f"RMSE – Women                   : {np.nanmean(rmse_f_l):6.1f}")
print(f"RMSE – Men                     : {np.nanmean(rmse_m_l):6.1f}")
print("="*65)


gender
Male      722
Female    722
Name: count, dtype: int64
gender
Male      665
Female    665
Name: count, dtype: int64

  PREPROCESSED MODEL – NO WEIGHTS (Repeated 5-fold CV, 50 folds) 
MSE               :  14129.3 ± 1423.8
RMSE              :    118.7 ± 6.0
MAE               :     91.5 ± 4.6
R²                :    0.625 ± 0.036

Mean predicted risk – Women  :  272.3
Mean predicted risk – Men      :  402.6
Mean residual (Y − Ŷ) – Women  :   -9.82
Mean residual (Y − Ŷ) – Men    :   +9.86
RMSE – Women                   :  100.2
RMSE – Men                     :  134.6


In [12]:
#2nd regresion model for data with pre-processing steps - weight 35% female

#loading data
data_with_prep = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/final_ihd_dataset_with_prepocessing.csv")    

#checking the row counts
print(data_with_prep['gender'].value_counts())

#deleting all the null values from the dataset
data_with_prep = data_with_prep.dropna(subset=['incidence_rate','year','cholesterol_level','diabetes_level','hypertension_level','smoking_level','obesity_level','alcohol_consumption', 'health_expenditure', 'weight_35f'])

#checking the row counts after deleting null values
print(data_with_prep['gender'].value_counts())

#extracting features and target variable
features = [
    'cholesterol_level',
    'diabetes_level', 
    'hypertension_level',
    'smoking_level',
    'obesity_level',
    'alcohol_consumption', 
    'health_expenditure',
    'year'          
]

X = sm.add_constant(data_with_prep[features])
y = data_with_prep['incidence_rate']
gender = data_with_prep['gender']
weights_35f = data_with_prep['weight_35f'].values   

# repeated 5-fold CV
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

#lists for results
mse_l, rmse_l, mae_l, r2_l = [], [], [], []
pred_f_l, pred_m_l = [], []
resid_f_l, resid_m_l = [], []
rmse_f_l, rmse_m_l = [], []


#training
for train_idx, test_idx in rkf.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    gender_te = gender.iloc[test_idx]
    w_tr = weights_35f[train_idx]
    
    #model fitting
    model = sm.WLS(y_tr, X_tr, weights=w_tr).fit()
    y_pred = model.predict(X_te)
    
    #performance
    mse_l.append(mean_squared_error(y_te, y_pred))
    rmse_l.append(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae_l.append(mean_absolute_error(y_te, y_pred))
    r2_l.append(r2_score(y_te, y_pred))

    #fairness
    f_mask = gender_te == 'Female'
    m_mask = gender_te == 'Male'
    
    pred_f_l.append(y_pred[f_mask].mean() if f_mask.any() else np.nan)
    pred_m_l.append(y_pred[m_mask].mean() if m_mask.any() else np.nan)
    resid_f_l.append((y_te[f_mask] - y_pred[f_mask]).mean() if f_mask.any() else np.nan)
    resid_m_l.append((y_te[m_mask] - y_pred[m_mask]).mean() if m_mask.any() else np.nan)
    rmse_f_l.append(np.sqrt(mean_squared_error(y_te[f_mask], y_pred[f_mask])) if f_mask.any() else np.nan)
    rmse_m_l.append(np.sqrt(mean_squared_error(y_te[m_mask], y_pred[m_mask])) if m_mask.any() else np.nan)

#results display
print("\n" + "="*65)
print("PREPROCESSED MODEL – WEIGHT 35% FEMALE (Repeated 5-fold CV, 50 folds)".center(65))
print("="*65)
print(f"MSE               : {np.mean(mse_l):8.1f} ± {np.std(mse_l):.1f}")
print(f"RMSE              : {np.mean(rmse_l):8.1f} ± {np.std(rmse_l):.1f}")
print(f"MAE               : {np.mean(mae_l):8.1f} ± {np.std(mae_l):.1f}")
print(f"R²                : {np.mean(r2_l):8.3f} ± {np.std(r2_l):.3f}")

print(f"\nMean predicted risk – Women  : {np.nanmean(pred_f_l):6.1f}")
print(f"Mean predicted risk – Men      : {np.nanmean(pred_m_l):6.1f}")
print(f"Mean residual (Y − Ŷ) – Women  : {np.nanmean(resid_f_l):+7.2f}")
print(f"Mean residual (Y − Ŷ) – Men    : {np.nanmean(resid_m_l):+7.2f}")
print(f"RMSE – Women                   : {np.nanmean(rmse_f_l):6.1f}")
print(f"RMSE – Men                     : {np.nanmean(rmse_m_l):6.1f}")
print("="*65)



gender
Male      722
Female    722
Name: count, dtype: int64
gender
Male      665
Female    665
Name: count, dtype: int64

PREPROCESSED MODEL – WEIGHT 35% FEMALE (Repeated 5-fold CV, 50 folds)
MSE               :  14139.0 ± 1420.4
RMSE              :    118.8 ± 6.0
MAE               :     91.5 ± 4.6
R²                :    0.625 ± 0.036

Mean predicted risk – Women  :  272.7
Mean predicted risk – Men      :  403.6
Mean residual (Y − Ŷ) – Women  :  -10.22
Mean residual (Y − Ŷ) – Men    :   +8.82
RMSE – Women                   :  101.0
RMSE – Men                     :  134.0


In [13]:
#3nd regresion model for data with pre-processing steps - weight 30% female

#loading data
data_with_prep = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/final_ihd_dataset_with_prepocessing.csv")    

#checking the row counts
print(data_with_prep['gender'].value_counts())

#deleting all the null values from the dataset
data_with_prep = data_with_prep.dropna(subset=['incidence_rate','year','cholesterol_level','diabetes_level','hypertension_level','smoking_level','obesity_level','alcohol_consumption', 'health_expenditure', 'weight_30f'])

#checking the row counts after deleting null values
print(data_with_prep['gender'].value_counts())

#extracting features and target variable
features = [
    'cholesterol_level',
    'diabetes_level', 
    'hypertension_level',
    'smoking_level',
    'obesity_level',
    'alcohol_consumption', 
    'health_expenditure',
    'year'          
]

X = sm.add_constant(data_with_prep[features])
y = data_with_prep['incidence_rate']
gender = data_with_prep['gender']
weights_30f = data_with_prep['weight_30f'].values   

# repeated 5-fold CV
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

#lists for results
mse_l, rmse_l, mae_l, r2_l = [], [], [], []
pred_f_l, pred_m_l = [], []
resid_f_l, resid_m_l = [], []
rmse_f_l, rmse_m_l = [], []


#training
for train_idx, test_idx in rkf.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    gender_te = gender.iloc[test_idx]
    w_tr = weights_30f[train_idx]
    
    #model fitting
    model = sm.WLS(y_tr, X_tr, weights=w_tr).fit()
    y_pred = model.predict(X_te)
    
    #performance
    mse_l.append(mean_squared_error(y_te, y_pred))
    rmse_l.append(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae_l.append(mean_absolute_error(y_te, y_pred))
    r2_l.append(r2_score(y_te, y_pred))

    #fairness
    f_mask = gender_te == 'Female'
    m_mask = gender_te == 'Male'
    
    pred_f_l.append(y_pred[f_mask].mean() if f_mask.any() else np.nan)
    pred_m_l.append(y_pred[m_mask].mean() if m_mask.any() else np.nan)
    resid_f_l.append((y_te[f_mask] - y_pred[f_mask]).mean() if f_mask.any() else np.nan)
    resid_m_l.append((y_te[m_mask] - y_pred[m_mask]).mean() if m_mask.any() else np.nan)
    rmse_f_l.append(np.sqrt(mean_squared_error(y_te[f_mask], y_pred[f_mask])) if f_mask.any() else np.nan)
    rmse_m_l.append(np.sqrt(mean_squared_error(y_te[m_mask], y_pred[m_mask])) if m_mask.any() else np.nan)

#results display
print("\n" + "="*65)
print("PREPROCESSED MODEL – WEIGHT 30% FEMALE (Repeated 5-fold CV, 50 folds)".center(65))
print("="*65)
print(f"MSE               : {np.mean(mse_l):8.1f} ± {np.std(mse_l):.1f}")
print(f"RMSE              : {np.mean(rmse_l):8.1f} ± {np.std(rmse_l):.1f}")
print(f"MAE               : {np.mean(mae_l):8.1f} ± {np.std(mae_l):.1f}")
print(f"R²                : {np.mean(r2_l):8.3f} ± {np.std(r2_l):.3f}")

print(f"\nMean predicted risk – Women  : {np.nanmean(pred_f_l):6.1f}")
print(f"Mean predicted risk – Men      : {np.nanmean(pred_m_l):6.1f}")
print(f"Mean residual (Y − Ŷ) – Women  : {np.nanmean(resid_f_l):+7.2f}")
print(f"Mean residual (Y − Ŷ) – Men    : {np.nanmean(resid_m_l):+7.2f}")
print(f"RMSE – Women                   : {np.nanmean(rmse_f_l):6.1f}")
print(f"RMSE – Men                     : {np.nanmean(rmse_m_l):6.1f}")
print("="*65)



gender
Male      722
Female    722
Name: count, dtype: int64
gender
Male      665
Female    665
Name: count, dtype: int64

PREPROCESSED MODEL – WEIGHT 30% FEMALE (Repeated 5-fold CV, 50 folds)
MSE               :  14175.8 ± 1417.1
RMSE              :    118.9 ± 6.0
MAE               :     91.7 ± 4.5
R²                :    0.624 ± 0.035

Mean predicted risk – Women  :  273.3
Mean predicted risk – Men      :  405.0
Mean residual (Y − Ŷ) – Women  :  -10.81
Mean residual (Y − Ŷ) – Men    :   +7.46
RMSE – Women                   :  102.3
RMSE – Men                     :  133.4


In [14]:
#4th regresion model for data with pre-processing steps - weight 25% female

#loading data
data_with_prep = pd.read_csv("C:/Users/user/Desktop/magisterka/sem_3/KU/data collection/final_ihd_dataset_with_prepocessing.csv")    

#checking the row counts
print(data_with_prep['gender'].value_counts())

#deleting all the null values from the dataset
data_with_prep = data_with_prep.dropna(subset=['incidence_rate','year','cholesterol_level','diabetes_level','hypertension_level','smoking_level','obesity_level','alcohol_consumption', 'health_expenditure', 'weight_25f'])

#checking the row counts after deleting null values
print(data_with_prep['gender'].value_counts())

#extracting features and target variable
features = [
    'cholesterol_level',
    'diabetes_level', 
    'hypertension_level',
    'smoking_level',
    'obesity_level',
    'alcohol_consumption', 
    'health_expenditure',
    'year'          
]

X = sm.add_constant(data_with_prep[features])
y = data_with_prep['incidence_rate']
gender = data_with_prep['gender']
weights_25f = data_with_prep['weight_25f'].values   

# repeated 5-fold CV
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

#lists for results
mse_l, rmse_l, mae_l, r2_l = [], [], [], []
pred_f_l, pred_m_l = [], []
resid_f_l, resid_m_l = [], []
rmse_f_l, rmse_m_l = [], []


#training
for train_idx, test_idx in rkf.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    gender_te = gender.iloc[test_idx]
    w_tr = weights_25f[train_idx]
    
    #model fitting
    model = sm.WLS(y_tr, X_tr, weights=w_tr).fit()
    y_pred = model.predict(X_te)
    
    #performance
    mse_l.append(mean_squared_error(y_te, y_pred))
    rmse_l.append(np.sqrt(mean_squared_error(y_te, y_pred)))
    mae_l.append(mean_absolute_error(y_te, y_pred))
    r2_l.append(r2_score(y_te, y_pred))

    #fairness
    f_mask = gender_te == 'Female'
    m_mask = gender_te == 'Male'
    
    pred_f_l.append(y_pred[f_mask].mean() if f_mask.any() else np.nan)
    pred_m_l.append(y_pred[m_mask].mean() if m_mask.any() else np.nan)
    resid_f_l.append((y_te[f_mask] - y_pred[f_mask]).mean() if f_mask.any() else np.nan)
    resid_m_l.append((y_te[m_mask] - y_pred[m_mask]).mean() if m_mask.any() else np.nan)
    rmse_f_l.append(np.sqrt(mean_squared_error(y_te[f_mask], y_pred[f_mask])) if f_mask.any() else np.nan)
    rmse_m_l.append(np.sqrt(mean_squared_error(y_te[m_mask], y_pred[m_mask])) if m_mask.any() else np.nan)

#results display
print("\n" + "="*65)
print("PREPROCESSED MODEL – WEIGHT 25% FEMALE (Repeated 5-fold CV, 50 folds)".center(65))
print("="*65)
print(f"MSE               : {np.mean(mse_l):8.1f} ± {np.std(mse_l):.1f}")
print(f"RMSE              : {np.mean(rmse_l):8.1f} ± {np.std(rmse_l):.1f}")
print(f"MAE               : {np.mean(mae_l):8.1f} ± {np.std(mae_l):.1f}")
print(f"R²                : {np.mean(r2_l):8.3f} ± {np.std(r2_l):.3f}")

print(f"\nMean predicted risk – Women  : {np.nanmean(pred_f_l):6.1f}")
print(f"Mean predicted risk – Men      : {np.nanmean(pred_m_l):6.1f}")
print(f"Mean residual (Y − Ŷ) – Women  : {np.nanmean(resid_f_l):+7.2f}")
print(f"Mean residual (Y − Ŷ) – Men    : {np.nanmean(resid_m_l):+7.2f}")
print(f"RMSE – Women                   : {np.nanmean(rmse_f_l):6.1f}")
print(f"RMSE – Men                     : {np.nanmean(rmse_m_l):6.1f}")
print("="*65)



gender
Male      722
Female    722
Name: count, dtype: int64
gender
Male      665
Female    665
Name: count, dtype: int64

PREPROCESSED MODEL – WEIGHT 25% FEMALE (Repeated 5-fold CV, 50 folds)
MSE               :  14245.9 ± 1416.0
RMSE              :    119.2 ± 5.9
MAE               :     92.0 ± 4.5
R²                :    0.622 ± 0.035

Mean predicted risk – Women  :  273.9
Mean predicted risk – Men      :  406.3
Mean residual (Y − Ŷ) – Women  :  -11.42
Mean residual (Y − Ŷ) – Men    :   +6.15
RMSE – Women                   :  103.8
RMSE – Men                     :  132.7
